In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [ ]:
calendar_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("Files/Bronze/calendar.csv")

sales_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("Files/Bronze/sales_train_validation.csv")

prices_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("Files/Bronze/sell_prices.csv")

In [ ]:
calendar_df.printSchema()

In [ ]:
sales_df.printSchema()

In [ ]:
prices_df.printSchema()

In [ ]:
display(calendar_df.limit(10))
display(prices_df.limit(10))
display(sales_df.limit(10))

In [ ]:
print("Calendar rows:", calendar_df.count())

print("Sales rows:", sales_df.count())

print("Price rows:", prices_df.count())

print("Calendar columns:", len(calendar_df.columns))

print("Sales columns:", len(sales_df.columns))

print("Price columns:", len(prices_df.columns))

print("Sales columns:")

print(sales_df.columns)

In [ ]:
from pyspark.sql.functions import col, sum

def null_report(df):
    return df.select([
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ])
    

In [ ]:
display(null_report(calendar_df))
display(null_report(prices_df))

In [ ]:
print(
    "Calendar duplicates:",
    calendar_df.count() - calendar_df.dropDuplicates().count()
)
print(
    "Price duplicates:",
    prices_df.count() - prices_df.dropDuplicates().count()
)

In [ ]:
calendar_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("Files/Bronze/calendar.csv")

sales_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("Files/Bronze/sales_train_validation.csv")

prices_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("Files/Bronze/sell_prices.csv")

In [ ]:
from pyspark.sql.functions import col, to_date, coalesce, lit

In [ ]:
silver_calendar = calendar_df \
    .withColumn("date", to_date(col("date"))) \
    .withColumn("event_name_1", coalesce(col("event_name_1"), lit("None"))) \
    .withColumn("event_type_1", coalesce(col("event_type_1"), lit("None"))) \
    .withColumn("event_name_2", coalesce(col("event_name_2"), lit("None"))) \
    .withColumn("event_type_2", coalesce(col("event_type_2"), lit("None"))) \
    .withColumn("snap_CA", col("snap_CA").cast("integer")) \
    .withColumn("snap_TX", col("snap_TX").cast("integer")) \
    .withColumn("snap_WI", col("snap_WI").cast("integer"))

In [ ]:
display(silver_calendar.limit(10))

In [ ]:
silver_calendar.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_calendar")

In [ ]:
silver_prices = prices_df \
    .withColumn("wm_yr_wk", col("wm_yr_wk").cast("integer")) \
    .withColumn("sell_price", col("sell_price").cast("double"))

In [ ]:
display(silver_prices.limit(10))

In [ ]:
silver_prices.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_prices")

In [ ]:
day_columns = [c for c in sales_df.columns if c.startswith("d_")]

len(day_columns)

In [ ]:
stack_expression = ", ".join(
    [f"'{c}', `{c}`" for c in day_columns]
)

In [ ]:
from pyspark.sql.functions import expr

base_columns = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

silver_sales = sales_df.select(
    *base_columns,
    expr(
        f"stack({len(day_columns)}, {stack_expression}) as (d, sales)"
    )
)

In [ ]:
display(silver_sales.limit(20))

In [ ]:
silver_sales = silver_sales \
    .withColumn("sales", col("sales").cast("integer"))

In [ ]:
display(silver_sales.limit(10))

In [ ]:
day_mapping = silver_calendar.select(
    "d",
    "date"
)

In [ ]:
display(day_mapping.limit(10))

In [ ]:
silver_sales_enriched = silver_sales.join(
    day_mapping,
    on="d",
    how="left"
)

In [ ]:
display(silver_sales_enriched.limit(10))

In [ ]:
silver_sales_enriched = silver_sales_enriched.drop("d")

In [ ]:
silver_sales_enriched = silver_sales_enriched.select(
    "date",
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "sales"
)

In [ ]:
display(silver_sales_enriched.limit(10))

In [ ]:
silver_sales_enriched.count()

In [ ]:
silver_sales_enriched.select(
    "date"
).agg(
    {"date": "min"}
).show()

silver_sales_enriched.select(
    "date"
).agg(
    {"date": "max"}
).show()

In [ ]:
silver_sales_enriched.filter(
    col("date").isNull()
).count()

In [ ]:
silver_sales_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_sales")

In [ ]:
spark.table("silver_sales").printSchema()

In [ ]:
display(
    spark.table("silver_sales").limit(10)
)

In [ ]:
print(
    "Silver Sales rows:",
    spark.table("silver_sales").count()
)

In [ ]:
print(
    "Silver Sales columns:",
    spark.table("silver_sales").columns
)

In [ ]:
silver_calendar = spark.table("silver_calendar")
silver_prices = spark.table("silver_prices")
silver_sales = spark.table("silver_sales")

In [ ]:
print("Calendar:", silver_calendar.count())
print("Prices:", silver_prices.count())
print("Sales:", silver_sales.count())

In [ ]:
gold_dim_date = silver_calendar.select(
    "date",
    "wm_yr_wk",
    "weekday",
    "wday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "snap_CA",
    "snap_TX",
    "snap_WI"
).withColumn(
    "day_of_month", dayofmonth(col("date"))
).withColumn(
    "day_of_week", dayofweek(col("date"))
).withColumn(
    "week_of_year", weekofyear(col("date"))
).withColumn(
    "quarter", ((col("month") - 1) / 3 + 1).cast("integer")
).withColumn(
    "is_event",
    when(
        (col("event_name_1") != "None") |
        (col("event_name_2") != "None"),
        1
    ).otherwise(0)
)

In [ ]:
display(gold_dim_date.limit(10))

In [ ]:
gold_dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_date")

In [ ]:
gold_dim_product = silver_sales.select(
    "item_id",
    "dept_id",
    "cat_id"
).dropDuplicates()

In [ ]:
display(gold_dim_product.limit(20))

In [ ]:
print("Product dimension rows:", gold_dim_product.count())

In [ ]:
gold_dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_product")

In [ ]:
gold_dim_store = silver_sales.select(
    "store_id",
    "state_id"
).dropDuplicates()

In [ ]:
display(gold_dim_store)

In [ ]:
print("Store dimension rows:", gold_dim_store.count())

In [ ]:
gold_dim_store.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_store")

In [ ]:
sales_with_week = silver_sales.join(
    silver_calendar.select(
        "date",
        "wm_yr_wk"
    ),
    on="date",
    how="left"
)

In [ ]:
display(sales_with_week.limit(10))

In [ ]:
sales_with_price = sales_with_week.join(
    silver_prices.select(
        "store_id",
        "item_id",
        "wm_yr_wk",
        "sell_price"
    ),
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

In [ ]:
display(sales_with_price.limit(10))

In [ ]:
from pyspark.sql.functions import round

gold_fact_sales = sales_with_price.withColumn(
    "revenue",
    round(
        col("sales") * col("sell_price"),
        2
    )
)

In [ ]:
gold_fact_sales = gold_fact_sales.select(
    "date",
    "item_id",
    "store_id",
    "state_id",
    "dept_id",
    "cat_id",
    "sales",
    "sell_price",
    "revenue"
)

In [ ]:
display(gold_fact_sales.limit(10))

In [ ]:
fact_count = gold_fact_sales.count()

print("Fact Sales rows:", fact_count)

In [ ]:
print(
    "Null dates:",
    gold_fact_sales.filter(
        col("date").isNull()
    ).count()
)

In [ ]:
print(
    "Negative sales:",
    gold_fact_sales.filter(
        col("sales") < 0
    ).count()
)

In [ ]:
print(
    "Sales with missing price:",
    gold_fact_sales.filter(
        (col("sales") > 0) &
        col("sell_price").isNull()
    ).count()
)

In [ ]:
gold_fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_fact_sales")

In [ ]:
gold_dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_date")

In [ ]:
silver_calendar = spark.table("silver_calendar")

In [ ]:
from pyspark.sql.functions import (
    col,
    dayofmonth,
    dayofweek,
    weekofyear,
    when
)

gold_dim_date = silver_calendar.select(
    "date",
    "wm_yr_wk",
    "weekday",
    "wday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "snap_CA",
    "snap_TX",
    "snap_WI"
).withColumn(
    "day_of_month",
    dayofmonth(col("date"))
).withColumn(
    "day_of_week",
    dayofweek(col("date"))
).withColumn(
    "week_of_year",
    weekofyear(col("date"))
).withColumn(
    "quarter",
    ((col("month") - 1) / 3 + 1).cast("integer")
).withColumn(
    "is_event",
    when(
        (col("event_name_1") != "None") |
        (col("event_name_2") != "None"),
        1
    ).otherwise(0)
)

In [3]:
silver_calendar = spark.table("silver_calendar")

StatementMeta(, d03f0798-cf3a-47a2-a659-dc70b2af29d5, 5, Finished, Available, Finished, False)

In [4]:
from pyspark.sql.functions import (
    col,
    dayofmonth,
    dayofweek,
    weekofyear,
    when
)

gold_dim_date = silver_calendar.select(
    "date",
    "wm_yr_wk",
    "weekday",
    "wday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "snap_CA",
    "snap_TX",
    "snap_WI"
).withColumn(
    "day_of_month",
    dayofmonth(col("date"))
).withColumn(
    "day_of_week",
    dayofweek(col("date"))
).withColumn(
    "week_of_year",
    weekofyear(col("date"))
).withColumn(
    "quarter",
    ((col("month") - 1) / 3 + 1).cast("integer")
).withColumn(
    "is_event",
    when(
        (col("event_name_1") != "None") |
        (col("event_name_2") != "None"),
        1
    ).otherwise(0)
)

StatementMeta(, d03f0798-cf3a-47a2-a659-dc70b2af29d5, 6, Finished, Available, Finished, False)

In [5]:
display(gold_dim_date.limit(10))

StatementMeta(, d03f0798-cf3a-47a2-a659-dc70b2af29d5, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3962f24d-071c-4e10-9627-407d75402b97)

In [6]:
gold_dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_date")

StatementMeta(, d03f0798-cf3a-47a2-a659-dc70b2af29d5, 8, Finished, Available, Finished, False)

In [ ]:
print("Gold Date:",
      spark.table("gold_dim_date").count())

print("Gold Product:",
      spark.table("gold_dim_product").count())

print("Gold Store:",
      spark.table("gold_dim_store").count())

print("Gold Fact Sales:",
      spark.table("gold_fact_sales").count())

In [ ]:
spark.table("gold_fact_sales").printSchema()

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_calendar = spark.read.table("silver_calendar")
silver_prices = spark.read.table("silver_prices")
silver_sales = spark.read.table("silver_sales")

print("silver_calendar")
silver_calendar.printSchema()

print("silver_prices")
silver_prices.printSchema()

print("silver_sales")
silver_sales.printSchema()

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 3, Finished, Available, Finished, False)

silver_calendar
root
 |-- date: date (nullable = true)
 |-- wm_yr_wk: integer (nullable = true)
 |-- weekday: string (nullable = true)
 |-- wday: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- d: string (nullable = true)
 |-- event_name_1: string (nullable = true)
 |-- event_type_1: string (nullable = true)
 |-- event_name_2: string (nullable = true)
 |-- event_type_2: string (nullable = true)
 |-- snap_CA: integer (nullable = true)
 |-- snap_TX: integer (nullable = true)
 |-- snap_WI: integer (nullable = true)

silver_prices
root
 |-- store_id: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- wm_yr_wk: integer (nullable = true)
 |-- sell_price: double (nullable = true)

silver_sales
root
 |-- date: date (nullable = true)
 |-- id: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- dept_id: string (nullable = true)
 |-- cat_id: string (nullable = true)
 |-- store_id: string (nullable = true)

In [3]:
gold_dim_date = (
    silver_calendar
    .select(
        "date",
        "wm_yr_wk",
        "weekday",
        "wday",
        "month",
        "year",
        "event_name_1",
        "event_type_1",
        "event_name_2",
        "event_type_2"
    )
    .dropDuplicates(["date"])
)

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 5, Finished, Available, Finished, False)

In [4]:
window_date = Window.orderBy("date")

gold_dim_date = (
    gold_dim_date
    .withColumn(
        "date_key",
        F.row_number().over(window_date)
    )
    .select(
        "date_key",
        "date",
        "wm_yr_wk",
        "weekday",
        "wday",
        "month",
        "year",
        "event_name_1",
        "event_type_1",
        "event_name_2",
        "event_type_2"
    )
)

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 6, Finished, Available, Finished, False)

In [5]:
display(gold_dim_date.limit(10))

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ef818eac-4245-40b1-bd98-59160fb5c17b)

In [6]:
gold_dim_date.write.mode("overwrite").format("delta").saveAsTable("gold_dim_date")

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 8, Finished, Available, Finished, False)

AnalysisException: [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: ab24a2a3-703f-435e-b7f9-4d5d3ab65241).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- date: date (nullable = true)
-- wm_yr_wk: integer (nullable = true)
-- weekday: string (nullable = true)
-- wday: integer (nullable = true)
-- month: integer (nullable = true)
-- year: integer (nullable = true)
-- event_name_1: string (nullable = true)
-- event_type_1: string (nullable = true)
-- event_name_2: string (nullable = true)
-- event_type_2: string (nullable = true)
-- snap_CA: integer (nullable = true)
-- snap_TX: integer (nullable = true)
-- snap_WI: integer (nullable = true)
-- day_of_month: integer (nullable =

In [7]:
gold_dim_date.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("gold_dim_date")

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 9, Finished, Available, Finished, False)

In [8]:
gold_dim_product = (
    silver_sales
    .select(
        "id",
        "item_id",
        "dept_id",
        "cat_id"
    )
    .dropDuplicates(["id"])
)

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 10, Finished, Available, Finished, False)

In [9]:
window_product = Window.orderBy("id")

gold_dim_product = (
    gold_dim_product
    .withColumn(
        "product_key",
        F.row_number().over(window_product)
    )
    .select(
        "product_key",
        "id",
        "item_id",
        "dept_id",
        "cat_id"
    )
)

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 11, Finished, Available, Finished, False)

In [10]:
display(gold_dim_product.limit(10))

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5d455a38-dbb4-41ae-ad46-30722ccee1d1)

In [11]:
gold_dim_product.write.mode("overwrite").format("delta").saveAsTable("gold_dim_product")

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 13, Finished, Available, Finished, False)

In [12]:
gold_dim_store = (
    silver_sales
    .select(
        "store_id",
        "state_id"
    )
    .dropDuplicates(["store_id"])
)

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 14, Finished, Available, Finished, False)

In [13]:
window_store = Window.orderBy("store_id")

gold_dim_store = (
    gold_dim_store
    .withColumn(
        "store_key",
        F.row_number().over(window_store)
    )
    .select(
        "store_key",
        "store_id",
        "state_id"
    )
)

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 15, Finished, Available, Finished, False)

In [14]:
gold_dim_store.write.mode("overwrite").format("delta").saveAsTable("gold_dim_store")

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 16, Finished, Available, Finished, False)

In [15]:
fact_sales = (
    silver_sales
    .join(
        gold_dim_date.select(
            "date",
            "date_key",
            "wm_yr_wk",
            "weekday",
            "month",
            "year",
            "event_name_1",
            "event_type_1",
            "event_name_2",
            "event_type_2"
        ),
        on="date",
        how="left"
    )
)

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 17, Finished, Available, Finished, False)

In [16]:
display(fact_sales.limit(10))

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e98b3008-20bd-43b5-b017-60d0b7616111)

In [17]:
fact_sales = (
    fact_sales
    .join(
        silver_prices.select(
            "store_id",
            "item_id",
            "wm_yr_wk",
            "sell_price"
        ),
        on=["store_id", "item_id", "wm_yr_wk"],
        how="left"
    )
)

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 19, Finished, Available, Finished, False)

In [18]:
display(
    fact_sales.select(
        "store_id",
        "item_id",
        "date",
        "wm_yr_wk",
        "sales",
        "sell_price"
    ).limit(20)
)

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5e542e60-9e05-407d-9e7d-9e32bf332194)

In [19]:
silver_prices.printSchema()


StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 21, Finished, Available, Finished, False)

root
 |-- store_id: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- wm_yr_wk: integer (nullable = true)
 |-- sell_price: double (nullable = true)



In [20]:
display(silver_prices.limit(20))


StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 21fca68b-04b3-4288-be09-4edbed896005)

In [21]:
silver_prices.filter(
    (F.col("store_id") == "CA_1") &
    (F.col("item_id") == "HOBBIES_1_001") &
    (F.col("wm_yr_wk") == 11101)
).show(truncate=False)

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 23, Finished, Available, Finished, False)

+--------+-------+--------+----------+
|store_id|item_id|wm_yr_wk|sell_price|
+--------+-------+--------+----------+
+--------+-------+--------+----------+



In [22]:
silver_prices.filter(
    (F.col("store_id") == "CA_1") &
    (F.col("item_id") == "HOBBIES_1_001")
).show(20, truncate=False)

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 24, Finished, Available, Finished, False)

+--------+-------------+--------+----------+
|store_id|item_id      |wm_yr_wk|sell_price|
+--------+-------------+--------+----------+
|CA_1    |HOBBIES_1_001|11325   |9.58      |
|CA_1    |HOBBIES_1_001|11326   |9.58      |
|CA_1    |HOBBIES_1_001|11327   |8.26      |
|CA_1    |HOBBIES_1_001|11328   |8.26      |
|CA_1    |HOBBIES_1_001|11329   |8.26      |
|CA_1    |HOBBIES_1_001|11330   |8.26      |
|CA_1    |HOBBIES_1_001|11331   |8.26      |
|CA_1    |HOBBIES_1_001|11332   |8.26      |
|CA_1    |HOBBIES_1_001|11333   |8.26      |
|CA_1    |HOBBIES_1_001|11334   |8.26      |
|CA_1    |HOBBIES_1_001|11335   |8.26      |
|CA_1    |HOBBIES_1_001|11336   |8.26      |
|CA_1    |HOBBIES_1_001|11337   |8.26      |
|CA_1    |HOBBIES_1_001|11338   |8.26      |
|CA_1    |HOBBIES_1_001|11339   |8.26      |
|CA_1    |HOBBIES_1_001|11340   |8.26      |
|CA_1    |HOBBIES_1_001|11341   |8.26      |
|CA_1    |HOBBIES_1_001|11342   |8.26      |
|CA_1    |HOBBIES_1_001|11343   |8.26      |
|CA_1    |

In [23]:
print("FACT:")
fact_sales.select(
    "store_id",
    "item_id",
    "wm_yr_wk"
).printSchema()

print("PRICES:")
silver_prices.select(
    "store_id",
    "item_id",
    "wm_yr_wk",
    "sell_price"
).printSchema()

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 25, Finished, Available, Finished, False)

FACT:
root
 |-- store_id: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- wm_yr_wk: integer (nullable = true)

PRICES:
root
 |-- store_id: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- wm_yr_wk: integer (nullable = true)
 |-- sell_price: double (nullable = true)



In [24]:
fact_sales.select(
    "store_id",
    "item_id",
    "date",
    "wm_yr_wk",
    "sales",
    "sell_price"
).filter(
    F.col("sell_price").isNotNull()
).show(20, truncate=False)

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 26, Finished, Available, Finished, False)

+--------+-----------+----------+--------+-----+----------+
|store_id|item_id    |date      |wm_yr_wk|sales|sell_price|
+--------+-----------+----------+--------+-----+----------+
|CA_1    |FOODS_1_001|2011-05-21|11117   |1    |2.0       |
|CA_1    |FOODS_1_001|2011-05-22|11117   |3    |2.0       |
|CA_1    |FOODS_1_001|2011-05-23|11117   |0    |2.0       |
|CA_1    |FOODS_1_001|2011-05-24|11117   |1    |2.0       |
|CA_1    |FOODS_1_001|2011-05-25|11117   |2    |2.0       |
|CA_1    |FOODS_1_001|2011-05-26|11117   |0    |2.0       |
|CA_1    |FOODS_1_001|2011-05-27|11117   |6    |2.0       |
|CA_1    |FOODS_1_001|2011-07-02|11123   |3    |2.0       |
|CA_1    |FOODS_1_001|2011-07-03|11123   |0    |2.0       |
|CA_1    |FOODS_1_001|2011-07-04|11123   |1    |2.0       |
|CA_1    |FOODS_1_001|2011-07-05|11123   |0    |2.0       |
|CA_1    |FOODS_1_001|2011-07-06|11123   |1    |2.0       |
|CA_1    |FOODS_1_001|2011-07-07|11123   |0    |2.0       |
|CA_1    |FOODS_1_001|2011-07-08|11123  

In [25]:
fact_sales.select(
    F.count("*").alias("total_rows"),
    F.count("sell_price").alias("rows_with_price"),
    F.sum(
        F.when(F.col("sell_price").isNull(), 1).otherwise(0)
    ).alias("rows_without_price")
).show()

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 27, Finished, Available, Finished, False)

+----------+---------------+------------------+
|total_rows|rows_with_price|rows_without_price|
+----------+---------------+------------------+
|  58327370|       46027957|          12299413|
+----------+---------------+------------------+



In [26]:
fact_sales = (
    fact_sales
    .withColumn(
        "revenue",
        F.round(
            F.col("sales") * F.col("sell_price"),
            2
        )
    )
)

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 28, Finished, Available, Finished, False)

In [27]:
display(
    fact_sales.select(
        "store_id",
        "item_id",
        "date",
        "sales",
        "sell_price",
        "revenue"
    ).limit(20)
)

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0b5a23e7-32fa-4cd7-9771-bce1637a3398)

In [28]:
fact_sales = (
    fact_sales
    .join(
        gold_dim_product.select(
            "product_key",
            "id"
        ),
        on="id",
        how="left"
    )
)

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 30, Finished, Available, Finished, False)

In [29]:
fact_sales = (
    fact_sales
    .join(
        gold_dim_store.select(
            "store_key",
            "store_id"
        ),
        on="store_id",
        how="left"
    )
)

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 31, Finished, Available, Finished, False)

In [30]:
fact_sales.select(
    F.count("*").alias("total_rows"),
    F.count("product_key").alias("rows_with_product_key"),
    F.count("store_key").alias("rows_with_store_key")
).show()

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 32, Finished, Available, Finished, False)

+----------+---------------------+-------------------+
|total_rows|rows_with_product_key|rows_with_store_key|
+----------+---------------------+-------------------+
|  58327370|             58327370|           58327370|
+----------+---------------------+-------------------+



In [31]:
gold_fact_sales = fact_sales.select(
    "date_key",
    "product_key",
    "store_key",
    "date",
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "sales",
    "sell_price",
    "revenue",
    "wm_yr_wk",
    "weekday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
)

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 33, Finished, Available, Finished, False)

In [32]:
gold_fact_sales.printSchema()

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 34, Finished, Available, Finished, False)

root
 |-- date_key: integer (nullable = true)
 |-- product_key: integer (nullable = true)
 |-- store_key: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- id: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- dept_id: string (nullable = true)
 |-- cat_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- state_id: string (nullable = true)
 |-- sales: integer (nullable = true)
 |-- sell_price: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- wm_yr_wk: integer (nullable = true)
 |-- weekday: string (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- event_name_1: string (nullable = true)
 |-- event_type_1: string (nullable = true)
 |-- event_name_2: string (nullable = true)
 |-- event_type_2: string (nullable = true)



In [33]:
display(gold_fact_sales.limit(20))

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c19d490b-47a1-4b97-9b19-80e3ee997355)

In [34]:
gold_fact_sales.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("gold_fact_sales")

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 36, Finished, Available, Finished, False)

In [35]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    SUM(sales) AS total_units_sold,
    SUM(revenue) AS total_revenue
FROM gold_fact_sales
""").show()

StatementMeta(, a6fb9088-192a-4d6d-93cc-0ab2d942301e, 37, Finished, Available, Finished, False)

+----------+----------------+--------------------+
|total_rows|total_units_sold|       total_revenue|
+----------+----------------+--------------------+
|  58327370|        65695409|1.8767657002118197E8|
+----------+----------------+--------------------+

